# 03 — Fine-tune a cross-encoder reranker (RD-12 / RD-13)

## Why this is the more interesting of the two

RD-12 measured off-the-shelf MS MARCO cross-encoders and rejected them — every
arm scored *below* plain retrieval. But it also **diagnosed why**, and the
diagnosis points at a fine-tuning target rather than a dead end:

> MS MARCO trains web-passage relevance, so the model ranks glosses by term
> overlap with the query — which is *echo relocated into the reranker*. A query
> is a **description** and a gloss is a **definition**: they are related by
> paraphrase, not overlap.

RD-16 then found the same cause at the other end of the pipeline: a QA-pretrained
bi-encoder lost 7.0 points. Two independent measurements, different
architectures, different stages, one conclusion — **what transfers is the
relation, not the size of the corpus.**

Nothing has yet tested a cross-encoder trained on the *right* relation.

## What RD-12 measured, so you do not repeat it

| arm | d=10 | d=25 | d=50 | d=100 | echo @100 |
|---|---|---|---|---|---|
| **no rerank** | **24.0** | — | — | — | **14.5%** |
| `ms-marco-MiniLM-L-6-v2`, gloss | 21.6 | 19.9 | 20.6 | 20.2 | 15.2% |
| `ms-marco-MiniLM-L-6-v2`, lemma-gloss | 23.3 | 22.6 | 22.3 | 21.6 | **21.4%** |
| `ms-marco-MiniLM-L-12-v2`, gloss | 24.4 | 22.6 | 23.0 | 22.0 | 15.0% |

Two things to carry forward. **Recall falls as shortlist depth rises**,
monotonically — that is what a near-uninformative ranking looks like. And the
lemma-gloss variant recovered ground only by driving echo from 14.5% to 21.4%,
which is why echo is scored here too.

## And the correction that changes the target

RD-21 measured the actual gaps: of the targets in the shortlist but not first,
**73.8% lose by a confident margin** (median 0.061), not a near-tie. So the job
is genuine rescoring, not tie-breaking.

That is *good* news for a cross-encoder specifically: it rescores from scratch
and does not inherit the bi-encoder's score surface. But it means a model that
only sharpens ties will do nothing.

In [ ]:
import sys, os
from pathlib import Path

# rdlib lives at training/rdlib; this notebook is at training/notebooks.
sys.path.insert(0, str(Path.cwd().parent))

# Cells are large and the darwin default lives under os.tmpdir(), which gets
# reaped. Point this somewhere durable and OUTSIDE the repo -- the working tree
# is in OneDrive, which would try to sync ~170 MB per cell.
os.environ.setdefault("EVAL_CELL_DIR", str(Path.home() / "rd_eval_cells"))

import rdlib
from rdlib import paths
print("repo      ", paths.REPO_ROOT)
print("cells     ", paths.cell_dir())

In [ ]:
from rdlib import parity
parity.report()

## The shortlist sidecar

`eval/runs/*.shortlist.jsonl` is the richest artifact in the repo for this work:
405 queries × 100 candidates, each carrying the synset key, **the gloss text
that was indexed**, its lemmas, the bi-encoder similarity, and RD-12's baseline
cross-encoder score.

**It is built from the eval set.** It is a dev set and a format template, and it
is never training data. Regenerate with `npm run eval:rerank` if missing.

In [ ]:
from rdlib import runs as R
from rdlib.evalset import load_eval_set, headline

ev = load_eval_set()
by_id = {r.id: r for r in ev}

try:
    shortlist = R.load_shortlist("rerank_ce6_p40_d100")
    print(f"{len(shortlist)} queries x {len(shortlist[0].candidates)} candidates")
except FileNotFoundError as exc:
    print(exc)
    shortlist = []

In [ ]:
# A worked failure: what the bi-encoder retrieved, and where the answer sat.
row = next(s for s in shortlist if s.id == "authored-0001")
answers = by_id[row.id].answers
print(f"query : {row.query}")
print(f"target: {row.target}   acceptable: {sorted(answers - {row.target})}\n")

for i, c in enumerate(row.candidates[:8], 1):
    hit = "<-- ANSWER" if any(l.lower() in answers for l in c["lemmas"]) else ""
    print(f"{i:>3}. sim={c['sim']:.3f}  {', '.join(c['lemmas'][:3]):<34} {hit}")
    print(f"      {c['gloss'][:92]}")

## Building training data — over disjoint queries

The positives and hard negatives have the same *shape* as the shortlist above,
but the queries must come from somewhere the benchmark has never seen.

The recipe: take (description, definition) pairs from `rdlib.pairs`, then mine
hard negatives by **actually retrieving** against the cell — a negative that the
bi-encoder ranks highly is the negative worth training on. Random negatives
teach almost nothing, because the model never sees them at inference.

In [ ]:
from rdlib import pairs as P
from rdlib.cells import load_cell
from rdlib.build import load_encoder, encode_texts
from rdlib.retrieval import search, _tie_break
from rdlib.evalset import assert_disjoint

train_pairs = P.pairs_example_to_gloss()
# Add the paraphrase recipe once the Kaikki dump is present -- it is the one
# that teaches the relation this notebook is about:
# train_pairs += P.pairs_wiktionary_paraphrase()

assert_disjoint([(p.query, p.target) for p in train_pairs], ev, check_targets=True)
print(f"PASS -- {len(train_pairs):,} pairs, disjoint from the benchmark")

In [ ]:
import numpy as np

cell = load_cell("rd22_gloss_ft")       # built in notebook 00
enc = load_encoder("franzclarin/ReverseDictionary")
tie = _tie_break(cell)

SAMPLE = 4000        # raise once the pipeline is proven
NEG_PER_POS = 4
sample = train_pairs[:SAMPLE]

qvecs = encode_texts(enc, [p.query for p in sample], progress=True)

ce_rows = []
for p, qv in zip(sample, qvecs):
    hits = search(cell, qv, 30, tie)
    negatives = [w for w, _ in hits if w.lower() != p.target.lower()][:NEG_PER_POS]
    ce_rows.append({"query": p.query, "passage": p.doc, "label": 1.0})
    # A negative needs the GLOSS of the wrong synset, not its lemma -- the
    # reranker scores (query, gloss) pairs, matching rerankText() in reranker.ts.
    for w in negatives:
        idx = cell.words.index(w) if w in cell.words else None
        if idx is not None:
            ce_rows.append({"query": p.query, "passage": cell.meta["words"][idx], "label": 0.0})

print(f"{len(ce_rows):,} labelled rows "
      f"({sum(r['label'] for r in ce_rows):,.0f} positive)")

> **Note on the cell above.** It mines negatives by *word*, which is the quick
> version. The honest version mines by **synset** and carries that synset's
> gloss text as the passage, because that is what the reranker actually scores
> at inference (`rerankText()` in `scripts/lib/reranker.ts`). Wire
> `cell.sense_keys` and a `{key: gloss}` map through before trusting a result —
> training on lemmas and inferring on glosses is its own quiet train/serve
> mismatch.

## Train

`CrossEncoderTrainer` with a binary objective. Start from the MS MARCO
checkpoint (it at least knows the *task* of scoring a pair) or from a plain
MiniLM to avoid inheriting the web-passage prior that RD-12 identified as the
problem. **Both are worth a run — that contrast is the actual experiment here.**

In [ ]:
import torch
from datasets import Dataset
from sentence_transformers.cross_encoder import (
    CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments,
)
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from rdlib.paths import ARTIFACTS_DIR

EXPERIMENT = "rd22_crossencoder_paraphrase"
OUT_DIR = ARTIFACTS_DIR / EXPERIMENT
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# "cross-encoder/ms-marco-MiniLM-L-6-v2" inherits the web-passage prior;
# "sentence-transformers/all-MiniLM-L6-v2" does not. Run both.
START_FROM = "cross-encoder/ms-marco-MiniLM-L-6-v2"

ds = Dataset.from_list(ce_rows).train_test_split(test_size=0.05, seed=20260830)
model = CrossEncoder(START_FROM, num_labels=1, device=DEVICE)

args = CrossEncoderTrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    eval_strategy="steps", eval_steps=200,
    logging_steps=50, seed=20260830, report_to=[],
)

CrossEncoderTrainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    loss=BinaryCrossEntropyLoss(model),
).train()

model.save_pretrained(str(OUT_DIR / "final"))
print(f"saved -> {OUT_DIR / 'final'}")

## Score it — by re-sorting the persisted shortlist

This is the cheap part, and RD-12 set it up well: a depth-*D* re-sort is a
**prefix** of the depth-100 scores, so one pass over the shortlist scores every
depth. Compare against `rerank_null` — retrieval alone at **24.0%**.

In [ ]:
from rdlib.metrics import QueryResult, score, compare
from rdlib.echo import content_tokens, echoes_query

reranker = CrossEncoder(str(OUT_DIR / "final"), device=DEVICE)
head_ids = {r.id for r in headline(ev)}
work = [s for s in shortlist if s.id in head_ids]

# One forward pass per (query, gloss) pair, at full depth.
all_scores = {}
for s in work:
    pairs_in = [(s.query, c["gloss"]) for c in s.candidates]
    all_scores[s.id] = reranker.predict(pairs_in, show_progress_bar=False)

def rerank_at(depth):
    out = []
    for s in work:
        cands, sc = s.candidates, all_scores[s.id]
        head = sorted(range(min(depth, len(cands))), key=lambda i: -sc[i])
        order = head + list(range(len(head), len(cands)))
        words, seen = [], set()
        for i in order:
            for lemma in cands[i]["lemmas"]:
                if lemma not in seen:
                    seen.add(lemma); words.append((lemma, float(sc[i])))
        out.append((s, words))
    return out

rows_ev = {r.id: r for r in headline(ev)}
print(f"{'depth':>6} {'lenient R@1':>12} {'strict R@1':>11} {'echo':>7}   vs 24.0%")
for depth in (10, 25, 50, 100):
    scored = []
    for s, words in rerank_at(depth):
        er = rows_ev[s.id]
        ws = [w for w, _ in words]
        li = next((i for i, w in enumerate(ws) if w.lower() in er.answers), -1)
        ti = next((i for i, w in enumerate(ws) if w.lower() == er.target.lower()), -1)
        toks = content_tokens(s.query)
        top10 = ws[:10]
        scored.append(QueryResult(
            id=s.id, query=s.query, target=s.target, source="authored",
            results=top10, similarities=[v for _, v in words[:10]],
            rank=None if ti < 0 else ti + 1,
            lenient_rank=None if li < 0 else li + 1,
            echo=sum(echoes_query(w, toks) for w in top10) / max(1, len(top10)),
            meta=er.meta))
    m = score(scored)
    print(f"{depth:>6} {m.lenient_recall1*100:>11.1f}% {m.recall1*100:>10.1f}% "
          f"{m.echo_rate*100:>6.1f}%   {m.lenient_recall1*100 - 24.0:+.1f}pp")

## Reading the result

- **Compare against 24.0%**, retrieval alone on this index. Every RD-12 arm came
  in below it.
- **Watch the depth curve.** If recall *falls* as depth rises, the ranking is
  near-uninformative — that was RD-12's clearest signal, and it is more
  diagnostic than any single cell.
- **Watch echo.** A gain that arrives with rising echo is overlap matching
  wearing a different hat.
- **§9a still binds**: under ~6pp is a null result. And RRF fusion already
  reached +1.8pp with no model at all, so that is the floor a trained reranker
  has to beat before it is interesting.

If it clears the bar, **RD-13** is the serving path — and it is blocked by its
own gate, which this notebook would be the evidence for. Note that serving a
cross-encoder means a second model in the `/api/lookup` bundle, which already
traces at 151.6 MB against a 250 MB limit. Check that before promising latency.